# An optimal first-order method for smooth and strongly convex composite optimization and its stationary limit

The following code uses PEPit [1] contains code for reproducing and verifying the main results of [2]

> [1] B. Goujaud, C. Moucer, F. Glineur, J. Hendrickx, A. Taylor, A. Dieuleveut.
> "PEPit: computer-assisted worst-case analyses of first-order optimization methods in Python,"
> Mathematical Programming Computation 16, 337–367, (2024).


> [2] Manu Upadhyaya, Daniel Berg Thomsen, Aymeric Dieuleveut, and Adrien Taylor.
> "An optimal first-order method for smooth and strongly convex composite optimization and its stationary limit," 2026.

## Authors

- [Manu Upadhyaya](https://manuupadhyaya.github.io/)
- [Daniel Berg Thomsen](https://bergthomsen.com/)
- [Aymeric Dieuleveut](http://www.cmap.polytechnique.fr/~aymeric.dieuleveut/)
- [Adrien Taylor](http://www.di.ens.fr/~ataylor/)

### Imports:

In [1]:
import numpy as np 

from math import sqrt

from PEPit import PEP
from PEPit.functions import SmoothStronglyConvexFunction
from PEPit.functions import ConvexFunction
from PEPit.primitive_steps import proximal_step

### Routines for the worst-case analyses of $n$ iterations of proxITEM and proxTMM

In [17]:
def wc_proximal_information_theoretic_exact_method(mu, L, n, verbose=1):
    """
    Worst-case guarantee of Prox-ITEM after n iterations.

    Problem class:
        minimize_x f(x) + h(x)
    where
        f is L-smooth and mu-strongly convex,
        h is convex.

    The method and its exact bound follow Algorithm 1 / Theorem 2.1 of the paper:
    
        ||z_n - x_*||^2 <= 1 / (1 + q A_n) ||x_0 - x_*||^2,
        
    where q = mu / L (inverse condition ratio) and A_n is generated by the Prox-ITEM recursion.
    """
    if not (0 < mu < L):
        raise ValueError("Require 0 < mu < L.")
    if n < 0:
        raise ValueError("Require n >= 0.")

    q = mu / L

    # ------------------------------------------------------------------
    # Instantiate PEP and declare the composite objective F = f + h
    # ------------------------------------------------------------------
    problem = PEP()

    f = problem.declare_function(SmoothStronglyConvexFunction, mu=mu, L=L)
    h = problem.declare_function(ConvexFunction)
    F = f + h

    # Optimal point x_*
    xs = F.stationary_point(name="xs")

    # Initial point x_0 = z_0
    x = problem.set_initial_point()
    z = x

    # Initial condition: ||x_0 - x_*||^2 <= 1
    problem.set_initial_condition((x - xs) ** 2 <= 1)

    # ------------------------------------------------------------------
    # Prox-ITEM coefficients (Algorithm 1)
    # ------------------------------------------------------------------
    A = [0.0]          # A[k] = A_k
    beta = []          # beta[k] = beta_k, k = 0,...,n-1
    delta = []         # delta[k] = delta_k, k = 0,...,n-1

    for k in range(n):
        Ak = A[-1]
        Ak1 = ((1 + q) * Ak + 2 * (1 + sqrt((1 + Ak) * (1 + q * Ak)))) / (1 - q) ** 2
        bk = Ak / ((1 - q) * Ak1)
        dk = sqrt(Ak1 / (1 + q * Ak1))

        A.append(Ak1)
        beta.append(bk)
        delta.append(dk)

    # ------------------------------------------------------------------
    # Run n Prox-ITEM iterations exactly as in Algorithm 1
    # ------------------------------------------------------------------
    for k in range(n):
        y = (1 - beta[k]) * z + beta[k] * x
        grad_y = f.gradient(y)

        z_bar = (1 - q * delta[k]) * z + q * delta[k] * y - (delta[k] / L) * grad_y
        z_next, _, _ = proximal_step(z_bar, h, delta[k] / L)

        x = y - (1 / L) * grad_y - (1 / delta[k]) * (z_bar - z_next)
        z = z_next

    # ------------------------------------------------------------------
    # Performance metric: final squared distance
    # ------------------------------------------------------------------
    problem.set_performance_metric((z - xs) ** 2)

    # Solve PEP
    pepit_tau = problem.solve(verbose=max(verbose, 0))

    # Exact theoretical guarantee from Theorem 2.1
    theoretical_tau = 1 / (1 + q * A[n])

    if verbose != -1:
        print("*** Prox-ITEM worst-case guarantee ***")
        print(f"\tPEPit guarantee:       ||z_n - x_*||^2 <= {pepit_tau:.12f} ||x_0 - x_*||^2")
        print(f"\tTheoretical guarantee: ||z_n - x_*||^2 <= {theoretical_tau:.12f} ||x_0 - x_*||^2")

    return pepit_tau, theoretical_tau


def wc_proximal_triple_momentum_method(mu, L, n, verbose=1):
    """
    Exact proof-based PEP for Prox-TMM, reindexed from a synthetic V_0^∞.

    We define V_0^∞ := V_1^∞ / (1 - sqrt(q))^2.
    
    Then the paper's bounds imply, for n >= 1:
    
        µ ||z_n - x_*||^2 <= (1 - sqrt(q))^(2n) V_0^∞.
        
    where q = mu / L (inverse condition ratio).
    """
    if not (0 < mu < L):
        raise ValueError("Require 0 < mu < L.")
    if n <= 1:
        raise ValueError("This experiment requires n > 1.")

    q = mu / L
    sq = sqrt(q)
    rho2 = (1 - sq) ** 2

    # ------------------------------------------------------------------
    # Instantiate PEP and declare the composite objective F = f + h
    # ------------------------------------------------------------------
    # Instantiate PEP
    problem = PEP()

    # Declare functions
    f = problem.declare_function(SmoothStronglyConvexFunction, mu=mu, L=L)
    g = problem.declare_function(ConvexFunction)
    F = f + g

    # Optimal point
    xs = F.stationary_point(name="xs")

    # ------------------------------------------------------------------
    # Initialization and initial Lyapunov value
    # ------------------------------------------------------------------
    # Initial point: x0 = z0
    x0 = problem.set_initial_point()
    x = x0
    z = x0
    
    # First step, needed to build V_1^∞
    y0 = (2 * sq / (1 + sq)) * z + ((1 - sq) / (1 + sq)) * x
    gy0 = f.gradient(y0)

    zbar1 = (1 - sq) * z + sq * y0 - (1 / (sq * L)) * gy0
    z1, _, _ = proximal_step(zbar1, g, 1 / (sq * L))

    x1 = y0 - (1 / L) * gy0 - sq * (zbar1 - z1)

    # Subgradient terms in the stationary Lyapunov
    sg1 = sq * L * (zbar1 - z1)   # in ∂g(z1)
    sgs = -f.gradient(xs)         # in ∂g(xs)
    
    # Interpolation inequalities
    def If(a, b):
        ga = f.gradient(a)
        gb = f.gradient(b)
        return (
            f(a) - f(b)
            - gb * (a - b)
            - (mu / 2) * (a - b) ** 2
            - (1 / (2 * (L - mu))) * (ga - gb - mu * (a - b)) ** 2
        )

    def Ig(a, b, s):
        return g(a) - g(b) - s * (a - b)

    # Paper's V_1^∞
    V1 = (
        (1 - q) * If(y0, xs)
        + q * Ig(xs, z1, sg1)
        + sq * (sq - 1) * Ig(z1, xs, sgs)
        + (1 / (2 * L)) * (sg1 - sgs) ** 2
        + mu * (z1 - xs) ** 2
    )
    
    # Exact proof-based normalization
    problem.set_initial_condition(V1 <= 1)
    
    # ------------------------------------------------------------------
    # Prox-TMM iteration (Algorithm 2)
    # ------------------------------------------------------------------
    # We already performed step 1 above
    x, z = x1, z1

    # Remaining steps up to z_n
    for _ in range(1, n):
        y = (2 * sq / (1 + sq)) * z + ((1 - sq) / (1 + sq)) * x
        gy = f.gradient(y)

        zbar = (1 - sq) * z + sq * y - (1 / (sq * L)) * gy
        z_next, _, _ = proximal_step(zbar, g, 1 / (sq * L))

        x = y - (1 / L) * gy - sq * (zbar - z_next)
        z = z_next

        

    # ------------------------------------------------------------------
    # Performance metric: final squared distance
    # ------------------------------------------------------------------
    problem.set_performance_metric(mu * (z - xs) ** 2)

    pepit_tau = problem.solve(verbose=max(verbose, 0))
    theoretical_tau = rho2 ** (n-1)

    if verbose != -1:
        print("*** Prox-TMM exact Lyapunov-based guarantee (from V_1^∞ == 1) ***")
        print(f"\tPEPit guarantee:       μ ||z_n - x_*||^2 <= {pepit_tau:.12f}")
        print(f"\tTheoretical guarantee: μ ||z_n - x_*||^2 <= {theoretical_tau:.12f}")

    return pepit_tau, theoretical_tau

### Run $n$ steps of proxITEM

In [18]:
mu, L = .1, 1
n = 10
verbose = 0

pepit_tau, theoretical_tau = wc_proximal_information_theoretic_exact_method(mu, L, n, verbose=verbose)

*** Prox-ITEM worst-case guarantee ***
	PEPit guarantee:       ||z_n - x_*||^2 <= 0.001025847109 ||x_0 - x_*||^2
	Theoretical guarantee: ||z_n - x_*||^2 <= 0.001025727228 ||x_0 - x_*||^2


### Run $n$ steps of proxTMM

In [19]:
mu, L = .1, 1
n = 10
verbose = 0

pepit_tau_tmm, theoretical_tau_tmm = wc_proximal_triple_momentum_method(mu, L, n, verbose=verbose)

*** Prox-TMM exact Lyapunov-based guarantee (from V_1^∞ == 1) ***
	PEPit guarantee:       μ ||z_n - x_*||^2 <= 0.001067594695
	Theoretical guarantee: μ ||z_n - x_*||^2 <= 0.001067594443
